# ฝึกและประเมินบน Mac
ใช้ kernel จาก .venv ของโครงการ ไม่มีผลรันที่สร้างขึ้น งานยาวให้เรียก scripts จาก Terminal; ไม่ผูกอายุ process กับ notebook

In [ ]:
from pathlib import Path
import os, sys, subprocess
root = Path.cwd()
if root.name == "notebooks": root = root.parent
os.chdir(root)
print(root, sys.executable)

In [ ]:
subprocess.run([sys.executable, "scripts/check_environment.py"], check=True)

## ตรวจข้อมูลและสิทธิก่อนฝึก
อ่าน data/source_manifest.json และ reports/training-data-audit.json ก่อนเลือกข้อมูลใหม่ ห้ามใช้ข้อมูลผู้ใช้หรือ software fixtures ฝึก

In [ ]:
import json
print(json.loads(Path("reports/training-data-audit.json").read_text())["counts"])

In [ ]:
subprocess.run([sys.executable, "scripts/prepare_training.py"], check=True)

## คำสั่งที่รันใน Terminal
```bash
.venv/bin/python scripts/train.py --config models/pilot.yaml
.venv/bin/python scripts/evaluate.py
.venv/bin/python scripts/run_demo.py
```
การรันซ้ำ pilot จะเขียนผลใน adapters/pilot; เปลี่ยน adapter_path หากต้องเก็บการทดลองเดิม

In [ ]:
base=json.loads(Path("reports/evaluation-base.json").read_text())
adapter=json.loads(Path("reports/evaluation-adapter.json").read_text())
print({"base_loss":base["heldout_loss"],"adapter_loss":adapter["heldout_loss"],"test_count":base["test_count"]})

## รอบหลายงาน fine-tuning v2
ข้อมูลฝึก 302 ตัวอย่าง แยก original English QA และ synthetic website commands ที่ผู้ใช้อนุญาต ผลจริงและข้อจำกัดอยู่ reports/finetune-v2-results.md ไม่ใช้คะแนน software/synthetic tests เป็น clinical accuracy. เซลล์นี้อ่านผลเท่านั้น ไม่เริ่มฝึกซ้ำหรือเขียนทับ adapter.


In [ ]:
import json
from pathlib import Path
training_v2 = json.loads(Path("reports/finetune-v2-training.json").read_text())
rollout_v2 = json.loads(Path("reports/finetune-v2-rollout.json").read_text())
print({"training_status": training_v2["status"], "steps": training_v2["config"]["iters"], "enabled_tasks": rollout_v2["enabled_tasks"]})


## รอบ specialist v3 และ runtime ปัจจุบัน
แยก adapter ตามงาน ข้อมูล test v2 ของ extraction/route เป็น training development ของ v3 แล้ว; ดูผล held-out v3 ในรายงานใหม่ ไม่เทียบเปอร์เซ็นต์กับ v2 โดยตรง.


In [ ]:
from services.model_runtime import runtime_status
print(runtime_status())
report_path = Path("reports/finetune-v3-training.json")
if report_path.exists():
    current = json.loads(report_path.read_text())
    print({"status": current.get("status"), "steps": current.get("config", {}).get("iters")})
